# 07 - Morphotope Postprocessing

Following source `06_morphotope_postprocessing.ipynb`:
Fix morphotopes so connected building components are not split across labels.
For each connected component, assign all buildings to the modal (non-noise) label.

In [1]:
import os
if os.path.basename(os.getcwd()).startswith("notebooks"):
    os.chdir(os.path.dirname(os.getcwd()))
print("Working dir:", os.getcwd())

import sys
sys.path.insert(0, "scripts")
from pipeline_config import load_city_config
load_city_config()

import numpy as np
import pandas as pd
from libpysal.graph import read_parquet
from pathlib import Path

OUTPUT_ROOT = os.environ.get("OUTPUT_ROOT", "outputs")
GRAPH_DIR = Path(OUTPUT_ROOT) / "03_graphs"
PRIMARY_CHARS_DIR = Path(OUTPUT_ROOT) / "05_primary_chars"
OUT = Path(OUTPUT_ROOT) / "06_morphotopes"

Working dir: C:\urban-form\urban_taxonomy
[pipeline_config] City config: config\canada.env


In [2]:
clusters = pd.read_parquet(OUT / "tessellation_labels_morphotopes.parquet")
bq1 = read_parquet(GRAPH_DIR / "building_graph.parquet")
X_train = pd.read_parquet(PRIMARY_CHARS_DIR / "primary_chars.parquet")

morph_labels = clusters["morphotope_label"]
print(f"{len(morph_labels):,} cells, {morph_labels.nunique()} morphotopes")

84,142 cells, 12568 morphotopes


In [3]:
def non_noise_mode(labels):
    """Return mode of non-noise labels; if all noise, return mode of those."""
    non_noise = labels[~labels.str.endswith('_-1')]
    if len(non_noise) > 0:
        return non_noise.mode().iloc[0]
    return labels.mode().iloc[0]

# Group by connected building components
comp_labels = bq1.component_labels
fixed = morph_labels.copy()

for comp_id, group in comp_labels.groupby(comp_labels):
    idx = group.index
    valid_idx = idx[idx.isin(morph_labels.index)]
    if len(valid_idx) > 1:
        modal = non_noise_mode(morph_labels.loc[valid_idx])
        fixed.loc[valid_idx] = modal

n_changed = (fixed != morph_labels).sum()
print(f"Fixed {n_changed:,} cells ({100*n_changed/len(morph_labels):.1f}%)")
print(f"Morphotopes: {morph_labels.nunique()} -> {fixed.nunique()}")

Fixed 6 cells (0.0%)
Morphotopes: 12568 -> 12568


In [4]:
# Regenerate morphotope aggregated stats with fixed labels
def percentile(n):
    def fn(x): return np.percentile(x, n)
    fn.__name__ = f'percentile_{n}'
    return fn

component_data = X_train.loc[fixed.index]
component_data = component_data.groupby(fixed.values).agg(
    [percentile(25), 'median', percentile(75), 'std', 'mean']
)
component_data[('Size', 'Size')] = (
    X_train.loc[fixed.index].groupby(fixed.values).size()
)

fixed.to_frame('morphotope_label').to_parquet(OUT / 'tessellation_labels_postprocessed.parquet')
component_data.to_parquet(OUT / 'data_morphotopes_postprocessed.parquet')
print(f"Saved postprocessed labels and data to {OUT}")

Saved postprocessed labels and data to outputs_canada\regions\4810\06_morphotopes
